# Corridas, artefactos y modelo de P2 en MLflow — MiBici

**Proyecto:** TC5061 · Equipo 52 · Pregunta P2 (clasificación) · Semana 6  
**Autor:** Roberto Robles · **Rama:** `feature/p2-mlflow-experimentos`  
**Antecedentes:** `2.1-rr-p2-particion-validacion.ipynb` (partición) y `equipo52_module/modeling/train_p2.py`.

**Propósito.** Aprender a leer desde Python lo que registra cada corrida de P2: parámetros, métricas y etiquetas; los artefactos en `plots/`, `data/` y `config/`; los conjuntos de datos (`log_input`), y el modelo con su firma. Con esto se arma la tabla comparativa del reporte.

**Requisito.** Tener encendido el servidor de MLflow (`mlflow server --backend-store-uri sqlite:///mlflow.db --host 127.0.0.1 --port 5000`) y al menos una corrida de P2 registrada. El notebook solo **lee**; no crea corridas.

In [ ]:
# Objetivo: conectarse al mismo servidor y experimento que usa train_p2.py.
# Resultado esperado: la dirección del servidor y el id del experimento mibici-equipo52.

import sys
from pathlib import Path

import mlflow
import pandas as pd
from IPython.display import Image, display

sys.path.insert(0, str(Path.cwd().parent))
from equipo52_module.config import MLFLOW_EXPERIMENT, MLFLOW_TRACKING_URI

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
cliente = mlflow.MlflowClient()
experimento = cliente.get_experiment_by_name(MLFLOW_EXPERIMENT)
print("Servidor:", MLFLOW_TRACKING_URI, "· experimento:", experimento.name, "· id:", experimento.experiment_id)

## 1. Buscar las corridas de P2 con las etiquetas

Las etiquetas `question`, `model_family` y `stage_of_work` sirven para filtrar. La misma expresión funciona en la caja de búsqueda de la interfaz de MLflow.

In [ ]:
# Objetivo: traer las corridas de P2 terminadas y ver sus etiquetas principales.
# Resultado esperado: una fila por corrida, con su etapa (baseline, search o candidate).

corridas = mlflow.search_runs([experimento.experiment_id],
                              filter_string="tags.question = 'p2' and attributes.status = 'FINISHED'",
                              order_by=["attributes.start_time DESC"])
corridas[["tags.mlflow.runName", "tags.stage_of_work", "tags.model_family", "params.periodo",
          "params.peso_fallas", "metrics.val_f1_fallas", "metrics.test_f1_fallas", "start_time"]]

## 2. Tabla comparativa contra el baseline

Es la base de la **Tabla 2** del reporte: métrica principal en validación y prueba, otras métricas y la mejora contra el baseline (F1 de fallas en prueba de `p2-base-completo`).

In [ ]:
# Objetivo: armar la tabla comparativa y calcular la mejora de cada corrida contra el baseline.
# Resultado esperado: el baseline con mejora 0 y cada candidato con su diferencia en puntos y en %.

base = corridas[corridas["tags.stage_of_work"] == "baseline"].iloc[0]
f1_base = base["metrics.test_f1_fallas"]
tabla = pd.DataFrame({
    "corrida": corridas["tags.mlflow.runName"],
    "etapa": corridas["tags.stage_of_work"],
    "familia": corridas["tags.model_family"],
    "periodo": corridas["params.periodo"],
    "peso_fallas": corridas.get("params.peso_fallas"),
    "learning_rate (hgb)": corridas.get("params.hgb_learning_rate"),
    "C (logística)": corridas.get("params.logistica_C"),
    "val F1 fallas": corridas["metrics.val_f1_fallas"].round(3),
    "test F1 fallas": corridas["metrics.test_f1_fallas"].round(3),
    "test recall sin bicicletas": corridas["metrics.test_recall_sin_bicicletas"].round(3),
    "test recall sin anclajes": corridas["metrics.test_recall_sin_anclajes"].round(3),
    "mejora (puntos)": (corridas["metrics.test_f1_fallas"] - f1_base).round(3),
    "mejora (%)": ((corridas["metrics.test_f1_fallas"] / f1_base - 1) * 100).round(1),
})
tabla

## 3. Artefactos de una corrida candidata

Cada corrida guarda sus archivos en subcarpetas: `plots/` (gráficas), `data/` (manifiesto y tablas) y `config/` (configuración con la que se ejecutó). Las corridas HGB guardan además el modelo en `modelo/`.

In [ ]:
# Objetivo: listar todos los artefactos de la corrida candidata más reciente.
# Resultado esperado: archivos en plots/, data/, config/ y modelo/.

candidata = corridas[corridas["tags.stage_of_work"] == "candidate"].iloc[0]
id_corrida = candidata["run_id"]
print("Corrida:", candidata["tags.mlflow.runName"], "·", id_corrida)


def listar(ruta=""):
    for archivo in cliente.list_artifacts(id_corrida, ruta):
        if archivo.is_dir:
            listar(archivo.path)
        else:
            print(f"  {archivo.path}  ({archivo.file_size:,} bytes)")


listar()

In [ ]:
# Objetivo: ver las dos gráficas de la corrida candidata.
# Resultado esperado: la matriz de confusión y la curva precisión-recall con el punto del modelo y del baseline.

for grafica in ["plots/p2_matriz_confusion_prueba.png", "plots/p2_precision_recall_prueba.png"]:
    display(Image(mlflow.artifacts.download_artifacts(run_id=id_corrida, artifact_path=grafica)))

## 4. Versión de los datos: `log_input`

En la sección *Datasets* de la corrida aparecen los conjuntos de entrenamiento y prueba. MLflow guarda su **descripción** (nombre, huella, origen, esquema y número de filas), no los datos.

In [ ]:
# Objetivo: leer los conjuntos registrados en la corrida.
# Resultado esperado: p2-entrenamiento-completo (training) y p2-prueba (testing), con su huella y número de filas.

for entrada in cliente.get_run(id_corrida).inputs.dataset_inputs:
    contexto = [t.value for t in entrada.tags if t.key == "mlflow.data.context"]
    print(entrada.dataset.name, contexto, "· huella", entrada.dataset.digest, "·", entrada.dataset.profile)

## 5. El modelo y su firma

La **firma** dice qué columnas y tipos recibe el modelo y qué devuelve. Si alguien le pasa columnas distintas, MLflow lo detiene antes de predecir.

In [ ]:
# Objetivo: cargar el modelo de la corrida y revisar su firma y su ejemplo de entrada.
# Resultado esperado: 13 variables de entrada y la clase predicha (0 normal, 1 sin bicicletas, 2 sin anclajes).

modelo = mlflow.pyfunc.load_model(f"runs:/{id_corrida}/modelo")
print(modelo.metadata.signature)
ejemplo = modelo.input_example
display(ejemplo)
print("Predicción para el ejemplo:", modelo.predict(ejemplo))

## 6. Interpretar la regresión logística

La regresión logística guarda sus coeficientes en `data/p2_coeficientes_logistica.csv`. Como las variables están estandarizadas, el tamaño de cada coeficiente indica cuánto empuja esa variable hacia cada clase: positivo la hace más probable, negativo menos. Las columnas `missingindicator_…` marcan las horas sin historial. HGB es más potente, pero no se puede leer así; esa diferencia sirve para el criterio de costo e interpretabilidad del reporte (Treveil et al., 2020, Tabla 4-2).

In [ ]:
# Objetivo: leer los coeficientes de la corrida logística más reciente, si ya existe.
# Resultado esperado: una fila por variable y una columna por clase, ordenadas por su peso hacia "sin bicicletas".

logisticas = corridas[corridas["tags.model_family"] == "logistica"]
if logisticas.empty:
    print("Todavía no hay corridas de regresión logística.")
else:
    ruta = mlflow.artifacts.download_artifacts(run_id=logisticas.iloc[0]["run_id"],
                                               artifact_path="data/p2_coeficientes_logistica.csv")
    coeficientes = pd.read_csv(ruta, index_col="variable")
    display(coeficientes.sort_values("sin_bicicletas", key=abs, ascending=False))

## 7. Próximos pasos

- **Paso F · búsqueda con nested runs:** una corrida padre con `stage_of_work = search` y una hija por combinación de hiperparámetros. Aquí agregaremos cómo leer el árbol padre-hijas.
- **Paso G · Model Registry:** registrar el modelo elegido como `mibici-equipo52-p2-riesgo` con alias `champion` y cargarlo con `models:/mibici-equipo52-p2-riesgo@champion`.